In [1]:
# Import Libraries
import pandas as pd
import exploration.explore as ex
import matplotlib as plot

In [75]:
pd.options.display.max_columns = 30

## Goals
- Create a combined dataset, and note any areas where joins will be difficult / will require logic
- Find a variable to use as our outcome - did a bill pass - and note different thresholds we might use
- Explore general shape of the data, e.g. how many bills are introduced, how many pass, breakdown by bill type
- Find and create clear features to use to inform 

### Goal 1: Create a combined datset (and note edge cases)

In [2]:
# Get all data associated with Bills
all_data = ex.get_all_datasets()

In [3]:
# Base data
bills = all_data["bills"]

bill_abstracts = all_data["bill_abstracts"]
bill_actions = all_data["actions"]
bills_sponsorships = all_data["bill_sponsorships"]
bill_organizations = all_data["organizations"]
bill_vote_counts = all_data["vote_counts"]
bill_votes = all_data["votes"]
bill_vote_people = all_data["vote_people"]

In [4]:
files = (bill_abstracts,
         bill_actions,
         bills_sponsorships,
         bills,
         bill_organizations,
         bill_vote_counts,
         bill_votes,
         bill_vote_people)

In [5]:
bills.shape  # (22783, 8) - the number of rows checks out

(22783, 8)

In [6]:
bills.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification
0,ocd-bill/c5de4bb9-edaf-4785-81a6-06768281eb76,SR 505,NO TAX DECOUPLING-FEDS,['resolution'],[],104th,Illinois,upper
1,ocd-bill/207b652d-33b8-490c-8e37-58c98bf08a3f,AM 1040067,APPOINT-JANICE MARIE GLENN,['appointment'],[],104th,Illinois,upper
2,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,HB 2686,REVENUE-LOCAL DISTRIBUTIONS,['bill'],[],104th,Illinois,lower
3,ocd-bill/41237f66-57f6-41bf-92eb-39fd09dd3570,AM 1040091,APPOINT-CHERITA ELLENS,['appointment'],[],104th,Illinois,upper
4,ocd-bill/31179116-d301-4232-b07d-0147b0c6cfd5,AM 1030583,APPOINT-MARY P. KENNELLY,['appointment'],[],104th,Illinois,upper


In [7]:
# Merge Bills and Bill Abstracts
df = bills.merge(bill_abstracts, how = "inner", left_on = "id", right_on = "bill_id")  # (22782, 12)

In [8]:
# Fix id column naming (not sure why it changed to "id_x")
df = df.rename(columns = {"id_x": "id"})

In [9]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note
0,ocd-bill/c5de4bb9-edaf-4785-81a6-06768281eb76,SR 505,NO TAX DECOUPLING-FEDS,['resolution'],[],104th,Illinois,upper,6f063feb-e08f-4c1c-80f5-835c157d99a4,ocd-bill/c5de4bb9-edaf-4785-81a6-06768281eb76,Respectfully urges the Governor of the State o...,NaN
1,ocd-bill/207b652d-33b8-490c-8e37-58c98bf08a3f,AM 1040067,APPOINT-JANICE MARIE GLENN,['appointment'],[],104th,Illinois,upper,c93a5c2b-9361-4e34-b266-c2f4a74d37b3,ocd-bill/207b652d-33b8-490c-8e37-58c98bf08a3f,Nominates Janice Marie Glenn as a Member of th...,NaN
2,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,HB 2686,REVENUE-LOCAL DISTRIBUTIONS,['bill'],[],104th,Illinois,lower,5794edbd-0521-4344-b53a-c3c6a7be36e5,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,Amends the State Finance Act and the State Rev...,NaN
3,ocd-bill/41237f66-57f6-41bf-92eb-39fd09dd3570,AM 1040091,APPOINT-CHERITA ELLENS,['appointment'],[],104th,Illinois,upper,7020e4ee-98d8-43ef-99e3-632a08d4a8f8,ocd-bill/41237f66-57f6-41bf-92eb-39fd09dd3570,Nominates Cherita Ellens as a Member of the La...,NaN
4,ocd-bill/31179116-d301-4232-b07d-0147b0c6cfd5,AM 1030583,APPOINT-MARY P. KENNELLY,['appointment'],[],104th,Illinois,upper,b6f22a3e-707c-4b18-9180-dd3b082f4417,ocd-bill/31179116-d301-4232-b07d-0147b0c6cfd5,Nominates Mary P. Kennelly as a Member of the ...,NaN


In [10]:
bill_actions.head()

,id,bill_id,organization_id,description,date,classification,order
0,09d3a35d-86f4-4c46-a730-a3a62ef1c60f,ocd-bill/bca5fea8-05bb-4dce-9011-aaf3a3bd7104,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. Charles Meier,2025-01-03,['filing'],0
1,586f33bc-e5c4-40df-91bb-2a13ea531eb8,ocd-bill/3d7bbce9-dbea-4454-bc70-dce43f5a0fa5,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. David Friess,2024-12-11,['filing'],0
2,c8c02f84-7fb4-4c82-b0e4-186dcf7e6421,ocd-bill/17f0335a-0183-4e2b-a50d-0610ef5c441b,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Adam M. Niemerg,2025-01-09,['filing'],0
3,0cf4e0e2-99e9-4809-a007-6bd4ec951a3e,ocd-bill/4cc9bb4a-8bb2-4091-8806-8730b4667f4d,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Chris Miller,2025-01-09,['filing'],0
4,15c22102-2cb2-4068-91ab-357fe56a862d,ocd-bill/50a41bc4-20fc-4137-b239-ebcfd1f46117,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. Kevin Schmidt,2025-01-03,['filing'],0


In [11]:
bill_actions["classification"].value_counts(normalize=True) * 100

classification
[]                                 58.608592
['reading-2']                       8.955676
['filing']                          8.483185
['reading-1']                       8.118580
['referral-committee']              6.377843
['amendment-introduction']          1.876577
['passage', 'reading-3']            1.441492
['passage']                         1.330063
['amendment-passage']               1.105236
['introduction']                    0.993413
['committee-passage-favorable']     0.644163
['executive-receipt']               0.601639
['became-law']                      0.597308
['executive-signature']             0.596914
['amendment-failure']               0.255933
['withdrawal']                      0.004725
['executive-veto']                  0.003544
['committee-failure']               0.003150
['failure', 'reading-3']            0.001181
['executive-veto-line-item']        0.000787
Name: proportion, dtype: float64

In [12]:
bill_actions.groupby("bill_id")["date"].count().sort_values(ascending=False).head(20)

bill_id
ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec    200
ocd-bill/86fa0a30-65be-4c01-908d-405fcb9fa294    198
ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d    187
ocd-bill/019153d8-a121-4414-bcbc-f9a4a2b4dd86    185
ocd-bill/dd3a42df-2f5a-4848-a2a7-0851def7b7fd    179
ocd-bill/dc8709ca-31cc-4498-b40c-7b7344789679    173
ocd-bill/a7200893-bcb1-4467-aeba-dd5f85b5862b    168
ocd-bill/bea2561c-bb19-48e7-867a-6d91880ead3b    166
ocd-bill/70380a50-000c-429b-a618-18c932ab7016    162
ocd-bill/e8e896d5-fd75-4a11-b6fd-fa1968bf411b    157
ocd-bill/2d13a3d5-4983-4214-9941-e014b1490677    155
ocd-bill/99f335f6-4ce1-46bd-9e3e-4095151752af    153
ocd-bill/85078383-95d5-4494-8b41-f7fec1b425d9    151
ocd-bill/d73a3b53-387a-4af2-be24-2f980c208ea8    149
ocd-bill/38f967ba-9287-4f34-841c-35e72e443c94    148
ocd-bill/c974da52-b390-4907-930a-aee431d609c0    142
ocd-bill/95ec692f-f395-49f0-9c65-b10e842e988f    141
ocd-bill/7eca898c-e95e-410a-8292-1edd8d3fd794    140
ocd-bill/41cb04fd-d282-4cda-b4dc-73a48

In [13]:
df[df["id"] == "ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec"]

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note
2580,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,HB 1312,POW MIA RECOGNITION DAY,['bill'],[],104th,Illinois,lower,7a0c8678-5622-4bbb-aceb-0f25f37e79b8,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,Amends the State Commemorative Dates Act. Prov...,NaN


In [14]:
bill_actions[bill_actions["bill_id"] == "ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec"].tail(8)

,id,bill_id,organization_id,description,date,classification,order
95310,7f5fe1dc-b272-4d78-86eb-679dea1d5336,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Added Co-Sponsor Rep. Angelica Guerrero-Cuellar,2025-10-31,[],192
95313,7051c387-2ca9-4604-b9b2-3c9af8e2ad96,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Motion to Reconsider Vote - Withdrawn Rep. Ann...,2025-10-31,[],193
95315,16c0dd2d-2bc1-445b-8be1-b8d9cfdaab06,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Passed Both Houses,2025-10-31,[],194
95316,c9d80445-d6ae-47cd-ab76-2632ccd85e85,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/20782ae5-cc22-4470-846b-fd978...,Added as Alternate Co-Sponsor Sen. Ram Villivalam,2025-11-07,[],195
95318,8d08908d-a5b0-4d2b-899b-f356483bb9b1,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Sent to the Governor,2025-11-25,['executive-receipt'],196
95320,264e7d27-e197-4db4-a276-717626454e91,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Governor Approved,2025-12-09,['executive-signature'],197
95322,4fcab682-1c1f-446d-948b-8e2ef53b22be,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Effective Date December 9, 2025",2025-12-09,[],198
95323,3bd09173-11fd-4183-9ff1-120e09534d73,ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Public Act . . . . . . . . . 104-0440,2025-12-09,['became-law'],199


In [15]:
num_bills_w_actions = bill_actions["bill_id"].nunique() # 22,783 Bills have unique actions
num_bills_w_actions

22783

In [16]:
mask = bill_actions["description"] == "Passed Both Houses"
num_bills_pass_both = bill_actions[mask]

In [17]:
num_bills_pass_both["bill_id"].nunique()

1537

In [18]:
mask2 = bill_actions["description"] == "Sent to the Governor"
num_bills_exec_received = bill_actions[mask2]

In [19]:
num_bills_exec_received["bill_id"].nunique()

1526

In [20]:
bill_actions["classification"].apply(lambda x: "executive-receipt" in x).sum()

np.int64(1528)

In [21]:
bill_actions["classification"].apply(lambda x: "became-law" in x).sum()

np.int64(1517)

In [22]:
bill_actions["classification"].apply(lambda x: "veto" in x).sum()

np.int64(11)

In [23]:
bill_actions.head(10)

,id,bill_id,organization_id,description,date,classification,order
0,09d3a35d-86f4-4c46-a730-a3a62ef1c60f,ocd-bill/bca5fea8-05bb-4dce-9011-aaf3a3bd7104,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. Charles Meier,2025-01-03,['filing'],0
1,586f33bc-e5c4-40df-91bb-2a13ea531eb8,ocd-bill/3d7bbce9-dbea-4454-bc70-dce43f5a0fa5,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. David Friess,2024-12-11,['filing'],0
2,c8c02f84-7fb4-4c82-b0e4-186dcf7e6421,ocd-bill/17f0335a-0183-4e2b-a50d-0610ef5c441b,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Adam M. Niemerg,2025-01-09,['filing'],0
3,0cf4e0e2-99e9-4809-a007-6bd4ec951a3e,ocd-bill/4cc9bb4a-8bb2-4091-8806-8730b4667f4d,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Chris Miller,2025-01-09,['filing'],0
4,15c22102-2cb2-4068-91ab-357fe56a862d,ocd-bill/50a41bc4-20fc-4137-b239-ebcfd1f46117,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. Kevin Schmidt,2025-01-03,['filing'],0
5,045ef274-ec02-43b0-aaf3-326bc7c4fa66,ocd-bill/5fe23d06-201f-4787-b3b2-444dc5fd0f32,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. La Shawn K. Ford,2024-12-02,['filing'],0
6,f7d01056-dc85-476f-aeca-b61e297511e5,ocd-bill/47fc7d00-9437-45ed-9f87-62fdd70ca231,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. John M. Cabello,2024-12-17,['filing'],0
7,7cb05b3e-c866-4cd0-9dcb-fc05be21202d,ocd-bill/7a05d290-7daa-4b45-8e9a-174237a893b3,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. Chris Miller,2025-01-06,['filing'],0
8,44879d84-a673-4943-9001-db303e52924f,ocd-bill/c388bc2b-356b-4505-9b0e-56ffc92afada,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. John M. Cabello,2024-12-17,['filing'],0
9,80156349-c0c3-4776-b1be-982f71587965,ocd-bill/6336144d-f7ce-4745-b927-6f8108370e12,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Prefiled with Clerk by Rep. Chris Miller,2024-12-04,['filing'],0


In [24]:
bill_actions.dtypes

id                   str
bill_id              str
organization_id      str
description          str
date                 str
classification       str
order              int64
dtype: object

In [25]:
bill_actions['classification'].value_counts()

classification
[]                                 148850
['reading-2']                       22745
['filing']                          21545
['reading-1']                       20619
['referral-committee']              16198
['amendment-introduction']           4766
['passage', 'reading-3']             3661
['passage']                          3378
['amendment-passage']                2807
['introduction']                     2523
['committee-passage-favorable']      1636
['executive-receipt']                1528
['became-law']                       1517
['executive-signature']              1516
['amendment-failure']                 650
['withdrawal']                         12
['executive-veto']                      9
['committee-failure']                   8
['failure', 'reading-3']                3
['executive-veto-line-item']            2
Name: count, dtype: int64

In [26]:
bill_actions["date"] = pd.to_datetime(bill_actions["date"])

### Create Summary Bill Actions Table

In [27]:
bill_actions.groupby("bill_id")["date"].agg(["max", "max"])

,max,max
bill_id,,
ocd-bill/00001671-dd8d-4867-9572-c8a250541072,2023-02-09,2023-02-09
ocd-bill/0000784d-cc84-47ac-912c-3d2ea037f93f,2025-05-31,2025-05-31
ocd-bill/0003da47-8a50-4198-8809-ff758e9df4bc,2025-04-11,2025-04-11
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71,2023-02-17,2023-02-17
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80,2023-03-10,2023-03-10
...,...,...
ocd-bill/fff7f8e7-1090-414c-b58f-da3d4dd4c1b7,2023-01-12,2023-01-12
ocd-bill/fff895cc-96a2-488a-b583-e73af4ae1bee,2024-02-09,2024-02-09
ocd-bill/fff9fcd7-659c-4aeb-a80a-bc256af05841,2023-03-31,2023-03-31


In [28]:
actions_summary = bill_actions.groupby("bill_id").agg(
    first_action = ("date", "min"),
    last_action = ("date", "max"),
    num_actions = ("id", "count"),
    passed_legislature = ("classification", lambda x: x.apply(lambda classif: "executive-receipt" in classif).any()),
    became_law = ("classification", lambda x: x.apply(lambda classif: "became-law" in classif).any()),
    vetoed = ("classification", lambda x: x.apply(lambda classif: "veto" in classif).any())
    ).sort_values(by="num_actions",ascending=False)

In [29]:
actions_summary["passed_legislature"].sum()

np.int64(1526)

In [30]:
actions_summary.head(5)

,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
bill_id,,,,,,
ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec,2025-01-14,2025-12-09,200,True,True,False
ocd-bill/86fa0a30-65be-4c01-908d-405fcb9fa294,2025-01-24,2025-07-01,198,True,True,False
ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,True,True,False
ocd-bill/019153d8-a121-4414-bcbc-f9a4a2b4dd86,2024-02-09,2024-07-10,185,True,True,False
ocd-bill/dd3a42df-2f5a-4848-a2a7-0851def7b7fd,2023-02-08,2023-12-08,179,True,True,False


In [31]:
df = df.merge(actions_summary, how="inner", left_on="id", right_on="bill_id")

In [32]:
df.head(5)

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
0,ocd-bill/c5de4bb9-edaf-4785-81a6-06768281eb76,SR 505,NO TAX DECOUPLING-FEDS,['resolution'],[],104th,Illinois,upper,6f063feb-e08f-4c1c-80f5-835c157d99a4,ocd-bill/c5de4bb9-edaf-4785-81a6-06768281eb76,Respectfully urges the Governor of the State o...,NaN,2025-10-30,2025-10-30,2,False,False,False
1,ocd-bill/207b652d-33b8-490c-8e37-58c98bf08a3f,AM 1040067,APPOINT-JANICE MARIE GLENN,['appointment'],[],104th,Illinois,upper,c93a5c2b-9361-4e34-b266-c2f4a74d37b3,ocd-bill/207b652d-33b8-490c-8e37-58c98bf08a3f,Nominates Janice Marie Glenn as a Member of th...,NaN,2025-02-25,2025-10-30,6,False,False,False
2,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,HB 2686,REVENUE-LOCAL DISTRIBUTIONS,['bill'],[],104th,Illinois,lower,5794edbd-0521-4344-b53a-c3c6a7be36e5,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,Amends the State Finance Act and the State Rev...,NaN,2025-02-04,2025-04-11,13,False,False,False
3,ocd-bill/41237f66-57f6-41bf-92eb-39fd09dd3570,AM 1040091,APPOINT-CHERITA ELLENS,['appointment'],[],104th,Illinois,upper,7020e4ee-98d8-43ef-99e3-632a08d4a8f8,ocd-bill/41237f66-57f6-41bf-92eb-39fd09dd3570,Nominates Cherita Ellens as a Member of the La...,NaN,2025-03-04,2025-10-30,9,False,False,False
4,ocd-bill/31179116-d301-4232-b07d-0147b0c6cfd5,AM 1030583,APPOINT-MARY P. KENNELLY,['appointment'],[],104th,Illinois,upper,b6f22a3e-707c-4b18-9180-dd3b082f4417,ocd-bill/31179116-d301-4232-b07d-0147b0c6cfd5,Nominates Mary P. Kennelly as a Member of the ...,NaN,2025-01-23,2025-10-30,10,False,False,False


In [33]:
df[df["passed_legislature"] == True].head(5)

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
11,ocd-bill/3d558c81-d66f-43be-a870-4447b07fd239,HB 1616,ORGAN DONOR LEAVE-PART-TIME,['bill'],[],104th,Illinois,lower,fbec426e-a3e9-405b-8468-8ca6b435b189,ocd-bill/3d558c81-d66f-43be-a870-4447b07fd239,Amends the Employee Blood and Organ Donation L...,NaN,2025-01-23,2025-08-15,60,True,True,False
13,ocd-bill/bdd805f3-8ec1-40a3-abcc-85aa7eb27081,SB 2510,$APPROPRIATIONS-VARIOUS,['bill'],[],104th,Illinois,upper,30b12f27-c56a-479c-b51f-5c1bb4655ee1,ocd-bill/bdd805f3-8ec1-40a3-abcc-85aa7eb27081,Appropriates $2 from the General Revenue Fund ...,NaN,2025-02-07,2025-10-29,56,True,False,True
47,ocd-bill/f5b25c86-9744-4ca6-b6cb-4f385ec336d0,HB 1082,MUNICIPALITIES-AUDITS,['bill'],[],104th,Illinois,lower,c0343469-d83c-4171-b891-a9d4187e2c3f,ocd-bill/f5b25c86-9744-4ca6-b6cb-4f385ec336d0,Amends the Illinois Municipal Auditing Law of ...,NaN,2025-01-02,2025-08-15,50,True,True,False
52,ocd-bill/c11a9345-15f8-48a5-b4bc-550a4a939b09,HB 1083,PROPERTY-GENDER NEUTRAL,['bill'],[],104th,Illinois,lower,583cd47d-0640-495d-808e-2daf9e72aa87,ocd-bill/c11a9345-15f8-48a5-b4bc-550a4a939b09,Amends the Illinois Religious Freedom Protecti...,NaN,2025-01-02,2025-08-01,45,True,True,False
84,ocd-bill/07c2fb09-57d7-4206-8261-a58716127e5d,SB 1555,SCH CD-SPEC ED ADVIS COUNCIL,['bill'],[],104th,Illinois,upper,fdaf8310-f136-4eac-b11a-33a6a7b5d67c,ocd-bill/07c2fb09-57d7-4206-8261-a58716127e5d,Amends the Children with Disabilities Article ...,NaN,2025-02-04,2025-08-01,31,True,True,False


In [34]:
df[df["title"].str.contains("GENDER NEUTRAL")]

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
52,ocd-bill/c11a9345-15f8-48a5-b4bc-550a4a939b09,HB 1083,PROPERTY-GENDER NEUTRAL,['bill'],[],104th,Illinois,lower,583cd47d-0640-495d-808e-2daf9e72aa87,ocd-bill/c11a9345-15f8-48a5-b4bc-550a4a939b09,Amends the Illinois Religious Freedom Protecti...,NaN,2025-01-02,2025-08-01,45,True,True,False
10337,ocd-bill/4d0b963b-abea-473c-8c3f-020a2aacb805,SB 2777,PROPERTY-GENDER NEUTRAL,['bill'],[],103rd,Illinois,upper,09da67e3-de74-4b43-bb20-8792abab870e,ocd-bill/4d0b963b-abea-473c-8c3f-020a2aacb805,Amends the Illinois Religious Freedom Protecti...,NaN,2024-01-17,2024-01-17,3,False,False,False
15832,ocd-bill/75a7eabd-aa28-488d-9a15-51e80f58fa2b,SJRCA 3,GENDER NEUTRAL-BILL OF RIGHTS,['constitutional amendment'],[],103rd,Illinois,upper,e3b5b0ec-faa0-49b5-98cb-c5ff37d310c4,ocd-bill/75a7eabd-aa28-488d-9a15-51e80f58fa2b,Proposes to amend the Bill of Rights Article o...,NaN,2023-02-28,2023-02-28,2,False,False,False


## Merging Bill Sponsors

In [35]:
bills_sponsorships.head()

,id,name,entity_type,organization_id,person_id,bill_id,primary,classification
0,49dbc51e-4631-4f7d-923c-445441a53982,Brad Stephens,person,NaN,ocd-person/49c0fac1-8cf0-48d2-8166-811afe9ac5c5,ocd-bill/74dc9755-20e3-42aa-b31b-d52a443a6a4d,True,primary
1,1fa5d460-30f0-483c-8f8f-43899c6f43b7,Adam M. Niemerg,person,NaN,ocd-person/9729745e-386d-4f4b-8d04-0e762dcd87b2,ocd-bill/03cc21de-6fa1-4e27-8f94-f3971424e90d,True,primary
2,13392b2b-6bc4-4d4f-8b0e-82c49c151f27,Lilian Jiménez,person,NaN,ocd-person/256a1090-5fef-4f0c-b016-263735573c77,ocd-bill/95382d39-2891-47c4-a6bf-e88d090750e0,True,primary
3,66a5a5c4-7694-407d-8282-1714943f5a82,Hoan Huynh,person,NaN,ocd-person/c7156dc3-31a6-4582-bf3a-b325c9dc868f,ocd-bill/e9205910-362d-4f00-b69f-1250cedee1c9,True,primary
4,7198e743-1bdd-4114-994a-0c0440215d14,Sonya M. Harper,person,NaN,ocd-person/8a39d16f-edbf-454b-8a77-33030fa9a8c2,ocd-bill/5ef5e32a-86e8-43d6-8f22-709aa09c81fe,True,primary


In [36]:
bills_sponsorships["entity_type"].unique()

<StringArray>
['person']
Length: 1, dtype: str

In [37]:
bills_sponsorships["organization_id"].unique()

array([nan])

In [38]:
bills_sponsorships.groupby("bill_id")["primary"].agg("sum").sort_values(ascending=False)

bill_id
ocd-bill/778a3168-e5e6-48c4-be44-b41c831ecc79    2
ocd-bill/778948f3-8274-42b8-bb8b-47c301579cfe    2
ocd-bill/d2c6e2a2-f8dc-4b39-9d23-be06eeee5079    2
ocd-bill/14690d1c-9635-4b1a-9698-2503373d323b    2
ocd-bill/eed6f278-c35d-4c7a-91e1-9305110c0a41    2
                                                ..
ocd-bill/5a77c57a-8bac-4f33-9a12-799444319c8c    1
ocd-bill/5a75eac2-ed56-4fa9-bad9-55cda9e201ed    1
ocd-bill/5a745ef4-ef58-4470-a9f3-6d4c50437f49    1
ocd-bill/5a6e94c3-d53f-4802-b69e-5cb2028331bb    1
ocd-bill/5a98b732-b262-46c8-9c39-2cf50d2e8965    1
Name: primary, Length: 22783, dtype: int64

In [39]:
 (bills_sponsorships.groupby("bill_id")["primary"].agg("sum") == 2).sum()  # 2,207 bills have 2 primary sponsors

np.int64(2207)

In [40]:
sponsors_summary = bills_sponsorships.groupby("bill_id").agg(
    num_sponsors = ("id", "count"))

In [41]:
primary_sponsors = bills_sponsorships["primary"] == True

In [42]:
primary_sponsors_summary = bills_sponsorships[primary_sponsors].groupby("bill_id").agg(
    primary_sponsor_1 = ("name", "min"),
    primary_sponsor_2 = ("name", "max"))

primary_sponsors_summary["num_primary"] = 2

In [43]:
# For bills with only 1 primary sponsor, clear out primary_sponsor_2
primary_sponsors_summary.loc[primary_sponsors_summary["primary_sponsor_2"] == primary_sponsors_summary["primary_sponsor_1"], "num_primary"] = 1
primary_sponsors_summary.loc[primary_sponsors_summary["primary_sponsor_2"] == primary_sponsors_summary["primary_sponsor_1"], "primary_sponsor_2"] = pd.NA

In [44]:
primary_sponsors_summary["primary_sponsor_2"].count()

np.int64(2206)

In [45]:
# Join two sponsor tables
sponsors_summary = sponsors_summary.merge(primary_sponsors_summary, how="inner", left_on="bill_id", right_on="bill_id")
sponsors_summary.head()

,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary
bill_id,,,,
ocd-bill/00001671-dd8d-4867-9572-c8a250541072,1,Steve Stadelman,NaN,1
ocd-bill/0000784d-cc84-47ac-912c-3d2ea037f93f,1,"Emanuel ""Chris"" Welch",NaN,1
ocd-bill/0003da47-8a50-4198-8809-ff758e9df4bc,1,Don Harmon,NaN,1
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71,1,"Marcus C. Evans, Jr.",NaN,1
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80,1,David Koehler,NaN,1


In [46]:
# Join full sponsors table with full bills table
df = df.merge(sponsors_summary, how="inner", left_on="id", right_on="bill_id")

In [47]:
df = df.drop(columns = ["id_y", "bill_id"])

In [48]:
# Adding an action timeframe column
df["first_to_last_action_time"] = df["last_action"] - df["first_action"]

In [76]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary,first_to_last_action_time
2,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,HB 2686,REVENUE-LOCAL DISTRIBUTIONS,['bill'],[],104th,Illinois,lower,Amends the State Finance Act and the State Rev...,NaN,2025-02-04,2025-04-11,13,False,False,False,2,"William ""Will"" Davis",NaN,1,66 days
5,ocd-bill/b6d316b6-ce6e-4c1f-a45f-3804fd599670,HB 3022,LIQUOR CONTROL-PROCEEDS,['bill'],[],104th,Illinois,lower,Amends the Liquor Control Act of 1934. Provide...,NaN,2025-02-06,2025-04-11,12,False,False,False,1,Will Guzzardi,NaN,1,64 days
8,ocd-bill/acb69ab0-bb26-4999-b1f5-339e04e9aadd,HB 4193,CANNABIS-R3 COMMITTEE,['bill'],[],104th,Illinois,lower,Amends the Cannabis Regulation and Tax Act. In...,NaN,2025-10-29,2025-10-29,3,False,False,False,1,"Maurice A. West, II",NaN,1,0 days
11,ocd-bill/3d558c81-d66f-43be-a870-4447b07fd239,HB 1616,ORGAN DONOR LEAVE-PART-TIME,['bill'],[],104th,Illinois,lower,Amends the Employee Blood and Organ Donation L...,NaN,2025-01-23,2025-08-15,60,True,True,False,31,Christopher Belt,Nabeela Syed,2,204 days
13,ocd-bill/bdd805f3-8ec1-40a3-abcc-85aa7eb27081,SB 2510,$APPROPRIATIONS-VARIOUS,['bill'],[],104th,Illinois,upper,Appropriates $2 from the General Revenue Fund ...,NaN,2025-02-07,2025-10-29,56,True,False,True,8,"Elgie R. Sims, Jr.","Emanuel ""Chris"" Welch",2,264 days


In [50]:
df.columns

Index(['id', 'identifier', 'title', 'classification', 'subject',
       'session_identifier', 'jurisdiction', 'organization_classification',
       'abstract', 'note', 'first_action', 'last_action', 'num_actions',
       'passed_legislature', 'became_law', 'vetoed', 'num_sponsors',
       'primary_sponsor_1', 'primary_sponsor_2', 'num_primary',
       'first_to_last_action_time'],
      dtype='str')

In [51]:
bill_votes.head()

,id,identifier,motion_text,motion_classification,start_date,result,organization_id,bill_id,bill_action_id,jurisdiction,session_identifier
0,ocd-vote/9cb9774e-bdfa-4f56-bf6e-a2d69d679424,NaN,Third Reading,[],2025-01-08,pass,ocd-organization/20782ae5-cc22-4470-846b-fd978...,ocd-bill/3f115748-efa0-407c-a981-47e9838becd4,NaN,Illinois,104th
1,ocd-vote/90ca71c5-5851-4f27-b352-06bac4008494,NaN,Third Reading,[],2025-01-09,pass,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,ocd-bill/439b73af-581a-4ad0-9397-2f8a5c34678a,NaN,Illinois,104th
2,ocd-vote/79bc5d99-cc6e-40d6-b5e5-507f52a3770b,NaN,Motion To Adopt,['passage'],2025-01-09,pass,ocd-organization/20782ae5-cc22-4470-846b-fd978...,ocd-bill/d787aa39-fab8-4a4d-bbb5-8b9886b074db,NaN,Illinois,104th
3,ocd-vote/83151d98-c3a9-4721-926e-7ef7c741d935,NaN,Judiciary,[],2025-01-29,pass,ocd-organization/20782ae5-cc22-4470-846b-fd978...,ocd-bill/0c8febdb-52cc-46f2-a231-10138aa1f474,NaN,Illinois,104th
4,ocd-vote/7ecbcafc-4514-4038-973b-a9b5db573d40,NaN,Judiciary,[],2025-01-29,pass,ocd-organization/20782ae5-cc22-4470-846b-fd978...,ocd-bill/7c285d3c-e374-4276-bead-ee49b289fe9f,NaN,Illinois,104th


In [52]:
bill_votes["motion_text"].value_counts().head(20)

motion_text
Executive                            5145
Third Reading                        4685
Executive Appointments                859
Judiciary                             315
Motion                                300
Concurrence                           282
Insurance                             251
Public Health                         240
State Government Administration       237
Education                             227
Higher Education                      185
State Government                      177
Labor & Commerce                      156
Human Services                        154
Concurrence, Amendment 1              146
Health and Human Services             145
Concurrence, Amendment 2              143
Energy & Environment                  138
Licensed Activities                   134
Elem Sec Ed: Adm., Lic. & Charter     132
Name: count, dtype: int64

In [54]:
df["first_to_last_action_time"].head(10)

0     0 days
1   247 days
2    66 days
3   240 days
4   280 days
5    64 days
6   226 days
7   247 days
8     0 days
9   246 days
Name: first_to_last_action_time, dtype: timedelta64[us]

### Looking more into actions to find other milestones to add

In [74]:
passed_legislature = df["passed_legislature"] == True
df[passed_legislature]["num_actions"].describe()  # The average bill that passes has 52 actions

count    1526.000000
mean       52.401048
std        24.070178
min        24.000000
25%        36.000000
50%        47.000000
75%        62.000000
max       200.000000
Name: num_actions, dtype: float64

In [73]:
df[passed_legislature]["first_to_last_action_time"].describe()

count                        1526
mean     202 days 11:13:45.688073
std       94 days 17:31:19.252895
min              59 days 00:00:00
25%             162 days 00:00:00
50%             182 days 00:00:00
75%             196 days 00:00:00
max             746 days 00:00:00
Name: first_to_last_action_time, dtype: object

In [77]:
df[passed_legislature]["num_sponsors"].describe()

count    1526.000000
mean       14.528834
std        15.583982
min         2.000000
25%         4.000000
50%         9.000000
75%        19.000000
max       134.000000
Name: num_sponsors, dtype: float64

In [56]:
df[df["passed_legislature"] == True].head(5)

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,abstract,note,...,last_action,num_actions,passed_legislature,became_law,vetoed,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary,first_to_last_action_time
11,ocd-bill/3d558c81-d66f-43be-a870-4447b07fd239,HB 1616,ORGAN DONOR LEAVE-PART-TIME,['bill'],[],104th,Illinois,lower,Amends the Employee Blood and Organ Donation L...,NaN,...,2025-08-15,60,True,True,False,31,Christopher Belt,Nabeela Syed,2,204 days
13,ocd-bill/bdd805f3-8ec1-40a3-abcc-85aa7eb27081,SB 2510,$APPROPRIATIONS-VARIOUS,['bill'],[],104th,Illinois,upper,Appropriates $2 from the General Revenue Fund ...,NaN,...,2025-10-29,56,True,False,True,8,"Elgie R. Sims, Jr.","Emanuel ""Chris"" Welch",2,264 days
47,ocd-bill/f5b25c86-9744-4ca6-b6cb-4f385ec336d0,HB 1082,MUNICIPALITIES-AUDITS,['bill'],[],104th,Illinois,lower,Amends the Illinois Municipal Auditing Law of ...,NaN,...,2025-08-15,50,True,True,False,18,Gregg Johnson,Sally J. Turner,2,225 days
52,ocd-bill/c11a9345-15f8-48a5-b4bc-550a4a939b09,HB 1083,PROPERTY-GENDER NEUTRAL,['bill'],[],104th,Illinois,lower,Amends the Illinois Religious Freedom Protecti...,NaN,...,2025-08-01,45,True,True,False,9,Daniel Didech,Sara Feigenholtz,2,211 days
84,ocd-bill/07c2fb09-57d7-4206-8261-a58716127e5d,SB 1555,SCH CD-SPEC ED ADVIS COUNCIL,['bill'],[],104th,Illinois,upper,Amends the Children with Disabilities Article ...,NaN,...,2025-08-01,31,True,True,False,3,Maura Hirschauer,Meg Loughran Cappel,2,178 days


## Starting Analysis

In [57]:
df.shape

(22782, 21)

In [58]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,abstract,note,...,last_action,num_actions,passed_legislature,became_law,vetoed,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary,first_to_last_action_time
0,ocd-bill/c5de4bb9-edaf-4785-81a6-06768281eb76,SR 505,NO TAX DECOUPLING-FEDS,['resolution'],[],104th,Illinois,upper,Respectfully urges the Governor of the State o...,NaN,...,2025-10-30,2,False,False,False,1,John F. Curran,NaN,1,0 days
1,ocd-bill/207b652d-33b8-490c-8e37-58c98bf08a3f,AM 1040067,APPOINT-JANICE MARIE GLENN,['appointment'],[],104th,Illinois,upper,Nominates Janice Marie Glenn as a Member of th...,NaN,...,2025-10-30,6,False,False,False,1,Laura M. Murphy,NaN,1,247 days
2,ocd-bill/2f334fe3-d0bd-4589-9e1c-80008d0eeea5,HB 2686,REVENUE-LOCAL DISTRIBUTIONS,['bill'],[],104th,Illinois,lower,Amends the State Finance Act and the State Rev...,NaN,...,2025-04-11,13,False,False,False,2,"William ""Will"" Davis",NaN,1,66 days
3,ocd-bill/41237f66-57f6-41bf-92eb-39fd09dd3570,AM 1040091,APPOINT-CHERITA ELLENS,['appointment'],[],104th,Illinois,upper,Nominates Cherita Ellens as a Member of the La...,NaN,...,2025-10-30,9,False,False,False,1,Laura M. Murphy,NaN,1,240 days
4,ocd-bill/31179116-d301-4232-b07d-0147b0c6cfd5,AM 1030583,APPOINT-MARY P. KENNELLY,['appointment'],[],104th,Illinois,upper,Nominates Mary P. Kennelly as a Member of the ...,NaN,...,2025-10-30,10,False,False,False,1,Laura M. Murphy,NaN,1,280 days


In [59]:
df.classification.value_counts(normalize=True) * 100  # 76.4% of bills are classified as Bill (pre-filtering)

classification
['bill']                        76.411202
['resolution']                  15.591256
['appointment']                  6.645597
['joint resolution']             1.057853
['constitutional amendment']     0.294092
Name: proportion, dtype: float64

In [60]:
df = df[df["classification"].apply(lambda x: "bill" in x)]

In [61]:
df.passed_legislature.mean() * 100  # 8.77% of bills passed the legislature

np.float64(8.766084558823529)

In [62]:
# TODO - Look into why no non-bills are passing. Need to use different mapping for actions?
df.groupby("classification")["passed_legislature"].agg("mean") * 100

classification
['bill']    8.766085
Name: passed_legislature, dtype: float64

In [63]:
df.groupby("session_identifier")["passed_legislature"].agg(["mean", "count"])

,mean,count
session_identifier,,
103rd,0.107294,9926
104th,0.061615,7482


In [64]:
df.groupby("organization_classification")["passed_legislature"].agg(["mean", "count"])

,mean,count
organization_classification,,
lower,0.077508,10515
upper,0.103148,6893


In [65]:
df.groupby(pd.cut(df["num_sponsors"], [0,1,  2, 5, 10, 15, 20, 25, 150]))["passed_legislature"].agg(["mean", "count"])

,mean,count
num_sponsors,,
"(0, 1]",0.000000,11794
"(1, 2]",0.085502,1614
"(2, 5]",0.242285,1523
"(5, 10]",0.315888,1070
"(10, 15]",0.410596,453
"(15, 20]",0.462541,307
"(20, 25]",0.500000,198
"(25, 150]",0.565702,449


In [66]:
df.groupby("num_primary")["passed_legislature"].agg(["mean", "count"])

,mean,count
num_primary,,
1,0.000000,15341
2,0.738268,2067


In [67]:
df.groupby(pd.cut(df["num_actions"], [0, 5, 10, 15, 20, 25, 30, 35, 45, 50, 200]))["passed_legislature"].agg(["mean", "count"])

,mean,count
num_actions,,
"(0, 5]",0.000000,6369
"(5, 10]",0.000000,6564
"(10, 15]",0.000000,1376
"(15, 20]",0.000000,556
"(20, 25]",0.058632,307
"(25, 30]",0.416201,358
"(30, 35]",0.615160,343
"(35, 45]",0.684512,523
"(45, 50]",0.761905,210


In [68]:
df.groupby(df["first_action"].dt.month)["passed_legislature"].agg(["mean", "count"])

,mean,count
first_action,,
1,0.076385,4857
2,0.114539,9368
3,0.000000,139
4,0.008547,117
5,0.018634,161
6,0.000000,9
7,0.030303,33
8,0.055556,36
9,0.014706,68


In [69]:
df.groupby("primary_sponsor_1")["passed_legislature"].agg(["mean", "count"]).sort_values("count", ascending = False)

,mean,count
primary_sponsor_1,,
"Emanuel ""Chris"" Welch",0.000000,1779
Don Harmon,0.019335,1293
Tony M. McCombie,0.000000,836
John F. Curran,0.005376,558
"Elgie R. Sims, Jr.",0.069892,372
...,...,...
Randy E. Frese,0.200000,5
Frances Ann Hurley,0.000000,4
John Egofske,0.000000,3


In [70]:
df.groupby("primary_sponsor_1")["passed_legislature"].agg(["mean", "count"]).sort_values("mean", ascending = False)

,mean,count
primary_sponsor_1,,
Ann Gillespie,1.000000,8
Cristina H. Pacione-Zayas,1.000000,1
Ann M. Williams,0.436620,71
Bob Morgan,0.415094,106
Jennifer Gong-Gershowitz,0.403226,62
...,...,...
Wayne A Rosenthal,0.000000,7
William E Hauter,0.000000,16
Willie Preston,0.000000,83
